# SatQuery AI — Division 2: Single-Image Remote-Sensing Intelligence
## Google Colab GPU Compute Pipeline & Training Runner

- **Division**: Division 2 (Single-Image Remote-Sensing Intelligence: VQA + Visual Grounding)
- **Owner**: Sruthi (`sruthi-270` / `rajamanurisruthi@gmail.com`)
- **Branch**: `feature/sruthi-single-image`
- **Target Model**: `google/paligemma-3b-pt-224` (Adapted via PEFT / LoRA rank=8)
- **Classification**: `[CONTROLLED BENCHMARK SUBSET EVALUATION — N=1,200 CORPUS / N=150 TEST]`

> **Instructions**: Run these cells sequentially in Google Colab with an active GPU runtime (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU` or `A100 GPU`).

### Step 1: GPU Compute Environment & Hardware Diagnostics

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:    {torch.version.cuda}")

### Step 2: Git Repository Checkout & Test Verification

In [ ]:
import os, sys
# Ensure clean single-level directory structure in Colab
if not os.path.exists('/content/SatQuery'):
    !git clone https://github.com/Lalith2007/SatQuery.git /content/SatQuery
%cd /content/SatQuery
!git fetch origin
!git checkout feature/sruthi-single-image
!git reset --hard origin/feature/sruthi-single-image

# Install project and async test dependencies
!pip install -q fastapi pydantic pydantic-settings tifffile pillow pytest pytest-asyncio pytest-cov psutil
!pip install -q -e .

# Generate initial adapter weights if needed for contract test
!python3 specialists/single_image/adaptation/generate_weights.py

# Set PYTHONPATH
if '/content/SatQuery' not in sys.path:
    sys.path.insert(0, '/content/SatQuery')

# Verify full test suite passes (67 tests)
!pytest tests/ -v --tb=short

### Step 3: Install ML & PEFT Dependencies

In [ ]:
!pip install -q transformers peft accelerate safetensors einops
import transformers, peft, accelerate
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"Accelerate:   {accelerate.__version__}")

### Step 4: Run Division 2 Environment Validation

In [ ]:
!python3 specialists/single_image/colab/gpu_validation.py

### Step 5: Execute LoRA Domain Adaptation Training on GPU (5 Epochs)

In [ ]:
!python3 specialists/single_image/adaptation/train_lora.py --device auto --epochs 5

### Step 6: Execute Evaluation & Scientific Reproducibility Audit (N=150 Held-Out Samples)

In [ ]:
!python3 specialists/single_image/evaluation/reproducibility.py

### Step 7: Export Reproducibility Manifest & Artifact Archive

In [ ]:
!python3 specialists/single_image/colab/reproducibility_manifest.py
!tar -czvf satquery_division2_adapter_package.tar.gz specialists/single_image/weights/ specialists/single_image/evaluation/ specialists/single_image/colab/
print("Artifact bundle generated: satquery_division2_adapter_package.tar.gz")